# Azure AI Foundry Agents via APIM + Application Insights Telemetry

This notebook demonstrates how to call **Azure AI Foundry hosted agents** through **Azure API Management (APIM)** and send token usage telemetry to **Application Insights**.

## Architecture

| Component | Role |
|-----------|------|
| **Azure AI Foundry** | Hosts the AI agents (e.g. WeatherAgent, HotelAgent, FinanceAgent) |
| **Azure API Management** | AI Gateway — routes, rate-limits, and traces agent requests |
| **Application Insights** | Collects token usage metrics and cost estimates via OpenTelemetry |

## What this notebook does

1. **Configures App Insights** — Sets up OpenTelemetry counters and a logger to send custom dimensions to the `traces` and `customMetrics` tables
2. **Calls agents via APIM** — Sends requests to Foundry-hosted agents through the APIM gateway endpoint
3. **Tracks token usage** — Extracts `input_tokens`, `output_tokens`, and `total_tokens` from each response and logs them with `agent_name`, `model`, and estimated `cost_usd` as custom dimensions

## Prerequisites

- **Azure CLI** authenticated (`az login`)
- **Python packages**: `httpx`, `azure-identity`, `azure-monitor-opentelemetry`, `opentelemetry-api`, `opentelemetry-sdk`
- **APIM subscription key** with access to the Foundry agent routes
- **App Insights connection string** (already embedded in the code below)

## Querying telemetry in App Insights

Go to your App Insights resource → **Logs** and run:

```kql
traces
| where message == "llm.usage"
| extend cd_raw = tostring(customDimensions["custom_dimensions"])
| extend cd = parse_json(replace_string(cd_raw, "'", "\""))
| extend
    agent_name        = tostring(cd["agent_name"]),
    model             = tostring(cd["model"]),
    operation_id      = tostring(cd["operation_id"]),
    prompt_tokens     = toint(cd["prompt_tokens"]),
    completion_tokens = toint(cd["completion_tokens"]),
    total_tokens      = toint(cd["total_tokens"]),
    cost_usd          = todouble(cd["cost_usd"])
| project timestamp, message, agent_name, model, operation_id, prompt_tokens, completion_tokens, total_tokens, cost_usd
| order by timestamp desc
```

> **Note:** Telemetry takes **2–5 minutes** to appear in App Insights after execution.

### Send Data and Metrics to AppInsight

In [6]:
! pip install azure-identity azure-monitor-query

In [ ]:
# ── Configuration — replace placeholders with your values ─────────────────────
APP_INSIGHTS_CONN = (
    "InstrumentationKey=<YOUR_INSTRUMENTATION_KEY>;"
    "IngestionEndpoint=<YOUR_INGESTION_ENDPOINT>;"
    "LiveEndpoint=<YOUR_LIVE_ENDPOINT>;"
    "ApplicationId=<YOUR_APPLICATION_ID>"
)

APP_INSIGHTS_RESOURCE_ID = (
    "/subscriptions/<YOUR_SUBSCRIPTION_ID>"
    "/resourceGroups/<YOUR_RESOURCE_GROUP>"
    "/providers/Microsoft.Insights/components/<YOUR_APP_INSIGHTS_NAME>"
)

APIM_BASE    = "<YOUR_APIM_GATEWAY_URL>"       # e.g. https://mygateway.azure-api.net
APIM_SUB_KEY = "<YOUR_APIM_SUBSCRIPTION_KEY>"
API_VERSION  = "2025-11-15-preview"

In [ ]:
import httpx
import logging
import atexit
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import metrics

# ── App Insights setup (runs once) ───────────────────────────────────────────
configure_azure_monitor(connection_string=APP_INSIGHTS_CONN)  # defined in config cell

# ── Logger → traces table (customDimensions always appear here) ──────────────
logger = logging.getLogger("agent.token.metrics")
logger.setLevel(logging.INFO)
logger.propagate = True  # 🔑 CRITICAL: must be True for OTel handler to receive logs

# ── Counters → customMetrics ─────────────────────────────────────────────────
meter = metrics.get_meter("agent.token.metrics")
prompt_counter     = meter.create_counter("llm.prompt_tokens",     unit="tokens", description="Input tokens consumed")
completion_counter = meter.create_counter("llm.completion_tokens", unit="tokens", description="Output tokens generated")
total_counter      = meter.create_counter("llm.total_tokens",      unit="tokens", description="Total tokens consumed")
cost_counter       = meter.create_counter("llm.request_cost_usd",  unit="USD",    description="Cumulative estimated cost")

MODEL_PRICING = {
    "gpt-4.1":     {"input": 0.002 / 1000,   "output": 0.008 / 1000},
    "gpt-4o":      {"input": 0.0025 / 1000,  "output": 0.010 / 1000},
    "gpt-4o-mini": {"input": 0.00015 / 1000, "output": 0.0006 / 1000},
}


def track_llm_usage(prompt_tokens, completion_tokens, model, agent_name, operation_id="unknown"):
    """Record token metrics + log with custom dimensions to App Insights."""
    total = prompt_tokens + completion_tokens
    pricing = MODEL_PRICING.get(model, {"input": 0.0, "output": 0.0})
    cost = (prompt_tokens * pricing["input"]) + (completion_tokens * pricing["output"])

    dims = {"agent_name": agent_name, "model": model, "operation_id": operation_id}

    # Counters → customMetrics
    prompt_counter.add(prompt_tokens, dims)
    completion_counter.add(completion_tokens, dims)
    total_counter.add(total, dims)
    cost_counter.add(cost, dims)

    # Logger → traces table (guaranteed customDimensions)
    logger.info(
        "llm.usage",
        extra={
            "custom_dimensions": {
                "agent_name":        agent_name,
                "model":             model,
                "operation_id":      operation_id,
                "prompt_tokens":     str(prompt_tokens),
                "completion_tokens": str(completion_tokens),
                "total_tokens":      str(total),
                "cost_usd":          str(round(cost, 6)),
            }
        },
    )

    print(f"  📊 Tracked → prompt={prompt_tokens}, completion={completion_tokens}, "
          f"total={total}, cost=${cost:.6f}")


# ──  Force flush metrics/logs on script exit ────────────────────────────────
def _shutdown():
    provider = metrics.get_meter_provider()
    if hasattr(provider, "shutdown"):
        provider.shutdown()
        print("  ✅ Metrics flushed to App Insights")

atexit.register(_shutdown)


# ── APIM config ──────────────────────────────────────────────────────────────
credential = DefaultAzureCredential()
_token_fn  = get_bearer_token_provider(credential, "https://ai.azure.com/.default")


def call_agent(agent_name: str, user_message: str) -> str:
    """Call an APIM-hosted agent, extract text, and send token metrics to App Insights."""
    token = _token_fn()

    r = httpx.post(
        f"{APIM_BASE}/foundry/models/{agent_name}",  # defined in config cell
        headers={
            "Authorization":             f"Bearer {token}",
            "Ocp-Apim-Subscription-Key": APIM_SUB_KEY,  # defined in config cell
            "Ocp-Apim-Trace":            "true",
            "Content-Type":              "application/json",
        },
        json={"model": "gpt-4.1", "input": user_message},
        params={"api-version": API_VERSION, "subscription-key": APIM_SUB_KEY},
        timeout=60,
    )
    r.raise_for_status()
    data = r.json()

    print("  trace location:", r.headers.get("Ocp-Apim-Trace-Location"))
    print("  raw response:", data)

    # ── Send token metrics to App Insights ───────────────────────────────────
    usage = data.get("usage", {})

    if usage:
        track_llm_usage(
            prompt_tokens=usage.get("input_tokens", 0),
            completion_tokens=usage.get("output_tokens", 0),
            model=data.get("model", "gpt-4.1"),
            agent_name=agent_name,
            operation_id="APIM-POST",
        )
    else:
        print("  ⚠️ No usage data in response")

    # ── Extract text from response ───────────────────────────────────────────
    for item in data.get("output", []):
        for part in item.get("content", []):
            if part.get("type") == "output_text":
                return part["text"]

    return str(data)  # fallback if no output_text found


# ── Entry point ──────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # ── Call WeatherAgent ─────────────────────────────────────────────────────
    weather = call_agent("WeatherAgent", "What is the weather in Seattle today?")
    print("\nWeatherAgent:", weather)

  trace location: None
  raw response: {'metadata': {}, 'top_logprobs': 0, 'temperature': 1, 'top_p': 1, 'service_tier': 'default', 'model': 'gpt-4.1', 'reasoning': {}, 'background': False, 'text': {'format': {'type': 'text'}, 'verbosity': 'medium'}, 'tools': [{'type': 'bing_grounding', 'bing_grounding': {'search_configurations': [{'project_connection_id': '/subscriptions/58fbb063-0114-4f12-9156-1578d11d36b6/resourceGroups/rg-msf-hosted-agents-10/providers/Microsoft.CognitiveServices/accounts/ccmshostedagents20/projects/ccmsprjhostedagents/connections/bingconnection02iv8wp5', 'market': 'en-us', 'set_lang': 'en', 'count': 5}]}}], 'tool_choice': 'auto', 'truncation': 'disabled', 'id': 'resp_0b2f34d8d138f6980169b17716c2f4819582943285cea32e30', 'object': 'response', 'status': 'completed', 'created_at': 1773238038, 'completed_at': 1773238043, 'error': None, 'incomplete_details': None, 'output': [{'type': 'bing_grounding_call', 'id': 'fc_0b2f34d8d138f6980169b1771915a48195b5f2d5461e3f6548', '

### Query Data

In [ ]:
from datetime import timedelta
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient

KQL = r"""
traces
| where message == "llm.usage"
| extend cd_raw = tostring(customDimensions["custom_dimensions"])
| extend cd = parse_json(replace_string(cd_raw, "'", "\""))
| extend
    agent_name        = tostring(cd["agent_name"]),
    model             = tostring(cd["model"]),
    operation_id      = tostring(cd["operation_id"]),
    prompt_tokens     = toint(cd["prompt_tokens"]),
    completion_tokens = toint(cd["completion_tokens"]),
    total_tokens      = toint(cd["total_tokens"]),
    cost_usd          = todouble(cd["cost_usd"])
| project timestamp, agent_name, model, operation_id,
          prompt_tokens, completion_tokens, total_tokens, cost_usd
| order by timestamp desc
"""

def query_logs():
    credential = DefaultAzureCredential()
    client = LogsQueryClient(credential)

    resp = client.query_resource(
        resource_id=APP_INSIGHTS_RESOURCE_ID,  # defined in config cell
        query=KQL,
        timespan=None,  # No time filter — returns all available data (up to 90-day retention)
    )

    if resp.status != "Success":
        raise RuntimeError(f"Query failed: {resp.status} - {getattr(resp, 'error', None)}")

    table = resp.tables[0]
    rows = [dict(zip(table.columns, r)) for r in table.rows]
    return rows

if __name__ == "__main__":
    rows = query_logs()
    if not rows:
        print("No telemetry found. Wait 2-5 min after running the agent cell and try again.")
    else:
        print(f"Found {len(rows)} records\n")
        print(f"{'Timestamp':<28} {'Agent':<16} {'Model':<12} {'Op ID':<12} "
              f"{'Prompt':>8} {'Completion':>11} {'Total':>8} {'Cost ($)':>10}")
        print("-" * 110)
        for r in rows[:20]:
            ts = str(r.get("timestamp", ""))[:19]
            print(f"{ts:<28} {r.get('agent_name',''):<16} {r.get('model',''):<12} "
                  f"{r.get('operation_id',''):<12} {r.get('prompt_tokens',0):>8} "
                  f"{r.get('completion_tokens',0):>11} {r.get('total_tokens',0):>8} "
                  f"{r.get('cost_usd',0):>10.6f}")

Found 11 records

Timestamp                    Agent            Model        Op ID          Prompt  Completion    Total   Cost ($)
--------------------------------------------------------------------------------------------------------------
2026-03-11 13:50:31          WeatherAgent     gpt-4.1      APIM-POST        5811          57     5868   0.012078
2026-03-11 03:12:06          WeatherAgent     gpt-4.1      APIM-POST        4547          56     4603   0.009542
2026-03-11 03:05:46          WeatherAgent     gpt-4.1      APIM-POST        4169          65     4234   0.008858
2026-03-10 22:50:32          WeatherAgent     gpt-4.1      APIM-POST        5880          49     5929   0.012152
2026-03-10 22:43:08          WeatherAgent     gpt-4.1      APIM-POST        5278          46     5324   0.010924
2026-03-10 22:19:59          WeatherAgent     gpt-4.1      APIM-POST        5662          56     5718   0.011772
2026-03-10 22:11:54          WeatherAgent     gpt-4.1      APIM-POST       10000

### Calculate Average and Total Metrics Per Agent

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient

KQL_AVG = r"""
traces
| where message == "llm.usage"
| extend cd_raw = tostring(customDimensions["custom_dimensions"])
| extend cd = parse_json(replace_string(cd_raw, "'", "\""))
| extend
    agent_name        = tostring(cd["agent_name"]),
    model             = tostring(cd["model"]),
    prompt_tokens     = toint(cd["prompt_tokens"]),
    completion_tokens = toint(cd["completion_tokens"]),
    total_tokens      = toint(cd["total_tokens"]),
    cost_usd          = todouble(cd["cost_usd"])
| summarize
    calls             = count(),
    avg_prompt_tokens        = avg(prompt_tokens),
    avg_completion_tokens    = avg(completion_tokens),
    avg_total_tokens         = avg(total_tokens),
    avg_cost_usd      = avg(cost_usd),
    total_cost_usd    = sum(cost_usd)
    by agent_name, model
| order by agent_name asc
"""

def query_avg_cost():
    credential = DefaultAzureCredential()
    client = LogsQueryClient(credential)

    resp = client.query_resource(
        resource_id=APP_INSIGHTS_RESOURCE_ID,  # defined in config cell
        query=KQL_AVG,
        timespan=None,
    )

    if resp.status != "Success":
        raise RuntimeError(f"Query failed: {resp.status} - {getattr(resp, 'error', None)}")

    table = resp.tables[0]
    rows = [dict(zip(table.columns, r)) for r in table.rows]
    return rows

if __name__ == "__main__":
    rows = query_avg_cost()
    if not rows:
        print("No telemetry found.")
    else:
        print(f"{'Agent':<16} {'Model':<12} {'Calls':>6} {'Avg Prompt':>11} "
              f"{'Avg Compl':>10} {'Avg Total':>10} {'Avg Cost':>10} {'Total Cost':>11}")
        print("-" * 95)
        for r in rows:
            print(f"{r.get('agent_name',''):<16} {r.get('model',''):<12} "
                  f"{r.get('calls',0):>6} {r.get('avg_prompt_tokens',0):>11.1f} "
                  f"{r.get('avg_completion_tokens',0):>10.1f} {r.get('avg_total_tokens',0):>10.1f} "
                  f"${r.get('avg_cost_usd',0):>9.6f} ${r.get('total_cost_usd',0):>10.6f}")

Agent            Model         Calls  Avg Prompt  Avg Compl  Avg Total   Avg Cost  Total Cost
-----------------------------------------------------------------------------------------------
WeatherAgent     gpt-4.1          12      5299.8       52.4     5352.2 $ 0.011019 $  0.132226
